In [13]:
# Loading the training data
import json
import pandas as pd

with open("../data/cuad/train_separate_questions.json") as f:
    train = json.load(f)

print("Data is a list of contracts. Each entry is one contract.")
print(train.keys(), '\n')
print("Number of training contracts:")
print(len(train["data"]))   

Data is a list of contracts. Each entry is one contract.
dict_keys(['version', 'data']) 

Number of training contracts:
408


In [14]:
#Loading the test data
import json

with open("../data/cuad/test.json") as f:
    test = json.load(f)

print("Test data keys:")
print(test.keys(), '\n')
print("Number of test contracts:")
print(len(test["data"]))

Test data keys:
dict_keys(['version', 'data']) 

Number of test contracts:
102


In [16]:
for name, data in [("TRAIN", train), ("TEST", test)]:
    doc = data["data"][0]          # doc = one contract
    qa = doc["paragraphs"][0]["qas"][0]

    print(name)
    print("  contracts:", len(data["data"]))
    print("  doc keys :", list(doc.keys()))
    print("  title    :", doc["title"][:50])
    print("  text len :", len(doc["paragraphs"][0]["context"]))
    print("  qas      :", len(doc["paragraphs"][0]["qas"]))
    print("  qa keys  :", list(qa.keys()))
    print("  answers  :", len(qa["answers"]))
    print()



TRAIN
  contracts: 408
  doc keys : ['title', 'paragraphs']
  title    : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMEN
  text len : 54290
  qas      : 68
  qa keys  : ['answers', 'id', 'question', 'is_impossible']
  answers  : 1

TEST
  contracts: 102
  doc keys : ['title', 'paragraphs']
  title    : LohaCompanyltd_20191209_F-1_EX-10.16_11917878_EX-1
  text len : 11475
  qas      : 41
  qa keys  : ['answers', 'id', 'question', 'is_impossible']
  answers  : 1



In [17]:
context = doc["paragraphs"][0]["context"]
print("Characters in the context:")
print(len(context))        
print(context[:2000])
print("Example Contract Text")

Characters in the context:
11475
Exhibit 10.16 SUPPLY CONTRACT Contract No: Date: The buyer/End-User: Shenzhen LOHAS Supply Chain Management Co., Ltd. ADD: Tel No. : Fax No. : The seller: ADD: The Contract is concluded and signed by the Buyer and Seller on , in Hong Kong. 1. General provisions 1.1 This is a framework agreement, the terms and conditions are applied to all purchase orders which signed by this agreement (hereinafter referred to as the "order"). 1.2 If the provisions of the agreement are inconsistent with the order, the order shall prevail. Not stated in order content will be subject to the provisions of agreement. Any modification, supplementary, give up should been written records, only to be valid by buyers and sellers authorized representative signature and confirmation, otherwise will be deemed invalid. 2. The agreement and order 2.1 During the validity term of this agreement, The buyer entrust SHENZHEN YICHANGTAI IMPORT AND EXPORT TRADE CO., LTD or SHENZHEN LEHEYUAN 

In [18]:
def explore(doc, label):
    qas = doc["paragraphs"][0]["qas"]

    # Train splits one category across several qas entries, test keeps them in one.
    # Grouping by category gives 41 either way.
    found = {}
    for qa in qas:
        category = qa["question"].split('"')[1]
        found.setdefault(category, [])
        for a in qa["answers"]:
            found[category].append(a["text"])

    print("=" * 80)
    print(label)
    print(doc["title"])
    print(f"{len(qas)} qas entries -> {len(found)} categories")
    print("=" * 80)

    for i, (category, answers) in enumerate(found.items(), 1):
        print(f"\n{i:2}. {category}  ({len(answers)} answer(s))")
        if not answers:
            print("     -- not in this contract --")
        for a in answers:
            text = " ".join(a.split())
            print(f"     > {text[:160]}{'...' if len(text) > 160 else ''}")


explore(train["data"][0], "TRAIN CONTRACT")
explore(test["data"][0], "TEST CONTRACT")


TRAIN CONTRACT
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
68 qas entries -> 41 categories

 1. Document Name  (1 answer(s))
     > DISTRIBUTOR AGREEMENT

 2. Parties  (5 answer(s))
     > Distributor
     > Electric City of Illinois L.L.C.
     > Electric City of Illinois LLC
     > Company
     > Electric City Corp.

 3. Agreement Date  (1 answer(s))
     > 7th day of September, 1999.

 4. Effective Date  (2 answer(s))
     > Unless earlier terminated otherwise provided therein, this Agreement, subject to the commencement date established in Section 1.3, shall be effective immediatel...
     > The term of this Agreement shall be ten (10) years (the "Term") which shall commence on the date upon which the Company delivers to Distributor the last Sample,...

 5. Expiration Date  (1 answer(s))
     > The term of this Agreement shall be ten (10) years (the "Term") which shall commence on the date upon which the Company delivers to Distributor the last Sample,...

 6. Renewal Term 

68 entries → 41 categories. The grouping is what reconciles train and test.
Most categories are empty. This contract has ~19 of 41. This is the class imbalance.
One category can have many answers (Parties has 5), so it's multi-label with multiple spans, not one answer per question.